In [1]:
# ================================================
# initialize
# ================================================
include("setup_notebook.jl")


MECH3620 Loading modules...
[OK] PyCall: C:/Users/kychandv/miniconda3/envs/mech3620/python.exe
TA_code path: c:\Users\kychandv\MECH3620\3620_project\MECH3620_Project\TA_code
TA_code exists: true
Added path: c:\Users\kychandv\MECH3620\3620_project\MECH3620_Project\TA_code


Importing functions...
[OK] Imported mech3620_models module
Available in mech3620_models:
[ERROR] Cannot import mech3620_models: MethodError(names, (PyObject <module 'mech3620_models' from 'c:\\Users\\kychandv\\MECH3620\\3620_project\\MECH3620_Project\\TA_code\\mech3620_models.py'>,), 0x00000000000097cb)
[OK] calc_thrust_lapse loaded
[OK] US_Standard_1976_Atmosphere loaded
[OK] calculate_time_to_climb loaded
[OK] calc_TOP_given_BFL_requirement loaded
[OK] Atmosphere instance created

Initialization complete!

Available functions/variables:
  ✓ calc_thrust_lapse
  ✓ US_Standard_1976_Atmosphere (class)
  ✓ atmosphere (instance)
  ✓ calc_TOP


"""
Only direct operating cost (DOC) need to cover for project

DOC = COC + FOC
"""

In [2]:
#==========================#
# Cash operating costs
#==========================#

function base_then_CDF(b_year, t_year)
    b_CEF = 5.17053+0.104981(b_year-2006)   # base CEF
    t_CEF = 5.17053+0.104981(t_year-2006)   # then CEF
    CEF = t_CEF/b_CEF   #effective CEF
    return b_CEF, t_CEF, CEF
end

base_then_CDF (generic function with 1 method)

In [3]:
#=========================#
# 1. Crew costs
#=========================#

function crew_costs(MTOW, t_b, CEF) #$/Flight 
    crew_cost = (482 + 0.590(MTOW/1000))*t_b * CEF
    return crew_cost
end

crew_costs (generic function with 1 method)

In [4]:
#=========================#
# 2. Attendants costs
#=========================#

function attendants_costs(n_attd, t_b, CEF) #$/Flight 
    attendants_cost = (78 * n_attd) * t_b * CEF
    return attendants_cost
end

attendants_costs (generic function with 1 method)

In [5]:
#=========================#
# 3. Fuel Cost
#=========================#

function fuel_costs(W_f, rho_f, P_f) #$/Flight 
    fuel_cost = 1.02 * W_f * (P_f / rho_f)
    return fuel_cost
end

fuel_costs (generic function with 1 method)

In [6]:
#=========================#
# 4. Oil cost
#=========================#

function oil_costs(rho_o, P_o, t_b, W_f)  # $/Flight
    W_o = 0.0125 * W_f * (t_b/100)
    oil_cost = 1.02 * W_o * (P_o / rho_o)
    return oil_cost
end

oil_costs (generic function with 1 method)

In [7]:
#=========================#
# 5. Landing fees
#=========================#

function landing_fees(MTOW, CEF)    # $/Flight
    landing_fee = 4.25 * (MTOW / 1000) * CEF
    return landing_fee
end

landing_fees (generic function with 1 method)

In [8]:
#=========================#
# 6. Navigation fees
#=========================#

function navigation_fees(MTOW, CEF)    # $/Flight
    navigation_fee = 68 * sqrt(MTOW/1000) * CEF
    return navigation_fee
end

navigation_fees (generic function with 1 method)

In [ ]:
#=========================#
# 7. Airframe maintenance costs
#=========================#

function airframe_maintenance_costs(MTOW, t_b, CEF, R_L)   # $/Flight
    P_aircraft = 10^(3.3191+0.8043*log10(MTOW))*CEF
    P_engine = 10^(2.3044+0.8858*log10(MTOW))*CEF
    P_airframe = P_aircraft - P_engine
    C_ML = 1.03 * (3 + 0.067 * MTOW/1000) * R_L
    C_MM = 1.03 * 30 * CEF + (0.79e-5) * P_airframe
    airframe_maintenance_cost = (C_ML + C_MM)* t_b
    return airframe_maintenance_cost, P_aircraft, P_engine, P_airframe
end

LoadError: syntax: { } vector syntax is discontinued around In[12]:6

In [ ]:
#=============================#
# 8. Engine maintenance costs
#=============================#
n_eng = 2

function engine_maintenance_costs(T_o, t_b, R_L)   # $/Flight
    C_ML = (0.645 + 0.05 * T_o/ 10^4)*(0.566 + 0.434/t_b) * R_L
    C_MM = (25+ 18*T_o/10^4) * (0.62 + 038/t_b) * CEF
    engine_maintenance_cost = n_eng * (C_ML + C_MM)* t_b
    return engine_maintenance_cost
end

In [ ]:
#============================#
# 9. Insurance cost
#============================#
IR_a = 0.02 # 2%
function insurance_cost(t_b, P_aircraft)   # $/Flight
    U_annual = 1.5e3 * (3.4546 * t_b + 2.994 - (12.289 *t_b^2 - 5.6626*t_b + 8.964)^0.5)#in hr
    insurance_cost = (IR_a * P_aircraft / U_annual) * t_b
    return insurance_cost, U_annual
end

In [ ]:
#============================#
# 10. Financing cost
#============================#
R_f = 0.05
function financing_cost(t_b, P_aircraft, U_annual, R_f)   # $/Flight
financing_cost = (R_f * P_aircraft / U_annual) * t_b
return financing_cost
end 

In [ ]:
#=============================#
# 11. Depreciation cost
#=============================#
K_depreciation = 0.3    # From note
n = 20  # years of operation form note
 function Depreciation_cost(P_aircraft, K_depreciation, t_b, U_annual)  #$/Flight
    depreciation_cost = ((1-K_depreciation) * P_aircraft / (n*U_annual)) * t_b
    return depreciation_cost
 end


In [ ]:
#=========================#
# 12. Registration fees
#=========================#

function registration_fees(MTOW, CEF)    # $/Flight
    registration_fee = (0.001+(10^(-8))*MTOW) * DOC
    return registration_fee
end

In [ ]:
b_year = 1993
t_year = 2026

#----------------------------------------------------------------
#unit transformation
#----------------------------------------------------------------
kgm3_to_lbgal = 0.00835 # lbs/gal
bbl_to_gal = 42 # gal/bbl
#-----------------------------------------------------------
CEF = base_then_CDF(b_year, t_year)[3]
b_CEF = base_then_CDF(b_year, t_year)[1]
t_CEF = base_then_CDF(b_year, t_year)[2]

MTOW = 33614.1 #kg, from 02
#1.--------------------------------------------------------------
t_b = 2.5      # flight block time, in hrs
crew_cost = crew_costs(MTOW, t_b, CEF)
#2.--------------------------------------------------------------
n_attd = 2      # no of flight attendants
attendants_cost = attendants_costs(n_attd, t_b, CEF)
#3.-------------------------------------------------------------
W_f = 11183.3 #kg, from 02, fuel weight
rho_f = 807.5 * kgm3_to_lbgal    # fuel density, lbs/gal, 775-840 g/L,take average here
P_f = 196.73/bbl_to_gal      # price of fuel USD/gal
fuel_cost = fuel_costs(W_f, rho_f, P_f)
#4.-------------------------------------------------------------
rho_o =  0.995* kgm3_to_lbgal*0.001   # oil density, lbs/gal, Eastman Turbo oil 25
P_o = 108.64      # price of oil USD/gal
oil_cost = oil_costs(rho_o, P_o, t_b, W_f)
#5.-------------------------------------------------------------
landing_fee = landing_fees(MTOW, CEF)
#6.-------------------------------------------------------------
navigation_fee = navigation_fees(MTOW, CEF)
#7.-------------------------------------------------------------
R_L = 28.72 # maintenance labor rate in USD/hr for the year of interest, Take the aircraft mechanics as reference
airframe_maintenance_cost, P_aircraft, P_engine, P_airframe = airframe_maintenance_costs(MTOW, t_b, CEF, R_L)
#8.-------------------------------------------------------------
engine_maintenance_cost = engine_maintenance_costs(T_o, t_b, R_L)
#9.-------------------------------------------------------------
insurance_cost, U_annual = insurance_cost(t_b, P_aircraft)
#10.-------------------------------------------------------------
financing_cost = financing_cost(t_b, P_aircraft, U_annual, R_f)
#11.-------------------------------------------------------------
depreciation_cost = Depreciation_cost(P_aircraft, K_depreciation, t_b, U_annual)


# Sum of all 1-11 cost-------------------------------------------------------------
DOC = crew_cost + attendants_cost + fuel_cost + oil_cost + landing_fee + navigation_fee + airframe_maintenance_cost + engine_maintenance_cost + insurance_cost + financing_cost + depreciation_cost


#12.-------------------------------------------------------------
registration_fee = registration_fees(MTOW, CEF, DOC)

# Total cost
total_cost = DOC + registration_fee

#Print results
println("Crew cost: $", crew_cost, "Unit: $/Flight")
println("Attendants cost: $", attendants_cost, "Unit: $/Flight")
println("Fuel cost: $", fuel_cost, "Unit: $/Flight")
println("Oil cost: $", oil_cost, "Unit: $/Flight")
println("Landing fee: $", landing_fee, "Unit: $/Flight")
println("Navigation fee: $", navigation_fee, "Unit: $/Flight")
println("Airframe maintenance cost: $", airframe_maintenance_cost, "Unit: $/Flight")
println("Engine maintenance cost: $", engine_maintenance_cost, "Unit: $/Flight")
println("Insurance cost: $", insurance_cost, "Unit: $/Flight")
println("Financing cost: $", financing_cost,    "Unit: $/Flight")
println("Depreciation cost: $", depreciation_cost, "Unit: $/Flight")
println("Direct operating cost (DOC): $", DOC, "Unit: $/Flight")
println("Registration fee: $", registration_fee, "Unit: $/Flight")
println("Total cost: $", total_cost, "Unit: $/Flight")